In [1]:
#!pip install transformers datasets torch scikit-learn

In [2]:
import pandas as pd
from datasets import Dataset

In [3]:
# Load CSV into pandas
df = pd.read_csv("huggingdata.csv")

In [4]:
df.head()

,text,label
0,I love this product!,positive
1,This is the worst experience ever.,negative
2,"Not bad, could be better.",neutral


In [5]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

In [6]:
# Split into train and test
dataset = dataset.train_test_split(test_size=0.2)

In [7]:
# Encode labels
from datasets import ClassLabel

# Collect labels from the full dataset
all_labels = list(dataset['train']['label']) + list(dataset['test']['label'])
unique_labels = sorted(set(all_labels))

class_labels = ClassLabel(num_classes=len(unique_labels), names=unique_labels)

def encode_labels(example):
    # strip spaces, lowercase to be safe
    example['label'] = class_labels.str2int(str(example['label']).strip().lower())
    return example

In [8]:
dataset = dataset.map(encode_labels)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [11]:
#Tokenization
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"]) # keep only input_ids, attention_mask, label
dataset.set_format("torch")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [15]:
#Load Model
from transformers import AutoModelForSequenceClassification

# Model
num_labels = len(all_labels)
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=num_labels,id2label=id2label,label2id=label2id)

NameError: name 'id2label' is not defined

In [13]:
# Training 
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

ImportError: cannot import name 'TFPreTrainedModel' from 'transformers' (C:\Users\Ashwinbarath\anaconda3\Lib\site-packages\transformers\__init__.py)